# FastFlow printer final reproduction

This notebook uses the same shared Anomalib FastFlow pipeline as the backbone notebooks. `load_frozen` only reads the six frozen results. For a new reproduction, run one config/seed at a time and commit its compact artifacts before running locked test or another training run.

In [ ]:
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")

import pandas as pd
import torch
from IPython.display import Image as NotebookImage, display
from PIL import Image

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (cwd, cwd.parent)
    if (candidate / "datasets").is_dir()
    and (candidate / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from fastflow_printer_pipeline import (
    PRINTER_CONFIGS,
    file_sha256,
    load_approved_manifest,
    load_experiment_result,
    run_experiment,
)

DATASET_ROOT = PROJECT_ROOT / "datasets" / "processed_printer_dataset_384"
MANIFEST_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "printer"
    / "dataset_v384_audit"
    / "printer_split_v2_final.csv"
)
SPLIT_CONFIG_PATH = PROJECT_ROOT / "configs" / "printer_split_v2_final.json"
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments" / "printer"

In [ ]:
# load_frozen: inspect all committed results without inference
# train_calibrate/test: execute one new config and seed
EXECUTION_MODE = "load_frozen"
CONFIG_NAME = "deit_base_distilled_384"
SEED = 42
TRY_NUMBER = 5
ALLOW_OVERWRITE = False
DETERMINISTIC = True
BOOTSTRAP_ITERATIONS = 2000

FROZEN_RUNS = [
    ("resnet18_384", 2, 42),
    ("resnet18_384", 3, 123),
    ("resnet18_384", 4, 2025),
    ("deit_base_distilled_384", 1, 42),
    ("deit_base_distilled_384", 2, 123),
    ("deit_base_distilled_384", 3, 2025),
]
DISPLAY_RUN = ("deit_base_distilled_384", 42)
EXPECTED_MANIFEST_SHA256 = (
    "aac844a06b740658ffcb756f033efd1722a0258f50b16c3e9dce0ae38d431317"
)

assert EXECUTION_MODE in {"load_frozen", "train_calibrate", "test"}
if CONFIG_NAME not in PRINTER_CONFIGS:
    raise KeyError(CONFIG_NAME)

In [ ]:
manifest, split_config = load_approved_manifest(
    DATASET_ROOT,
    MANIFEST_PATH,
    SPLIT_CONFIG_PATH,
    verify_hashes=True,
)
manifest_sha256 = file_sha256(MANIFEST_PATH)
assert manifest_sha256 == EXPECTED_MANIFEST_SHA256

split_summary = (
    manifest.groupby(["split", "class_name"])
    .agg(
        images=("path", "size"),
        source_images=("source_group", "nunique"),
        objects=("object_group", "nunique"),
    )
    .reset_index()
)
print(f"split_version={split_config['version']}")
print(f"manifest_sha256={manifest_sha256}")
display(split_summary)

In [ ]:
if EXECUTION_MODE == "load_frozen":
    results = [
        load_experiment_result(
            EXPERIMENTS_ROOT, PRINTER_CONFIGS[config_name], try_number, seed
        )
        for config_name, try_number, seed in FROZEN_RUNS
    ]
else:
    config = PRINTER_CONFIGS[CONFIG_NAME]
    result = run_experiment(
        config=config,
        seed=SEED,
        dataset_root=DATASET_ROOT,
        manifest_path=MANIFEST_PATH,
        split_config_path=SPLIT_CONFIG_PATH,
        experiments_root=EXPERIMENTS_ROOT,
        try_number=TRY_NUMBER,
        run_training=EXECUTION_MODE == "train_calibrate",
        run_calibration=EXECUTION_MODE == "train_calibrate",
        run_test=EXECUTION_MODE == "test",
        allow_overwrite=ALLOW_OVERWRITE,
        deterministic=DETERMINISTIC,
        verify_manifest_hashes=False,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    )
    results = [result]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
summary_df = pd.DataFrame(results)

In [ ]:
test_columns = [
    "test_source_group_balanced_tile_roc_auc",
    "test_primary_roc_auc_ci_low",
    "test_primary_roc_auc_ci_high",
    "test_object_roc_auc",
    "test_source_image_roc_auc",
    "test_source_group_balanced_tile_balanced_accuracy",
    "test_source_group_balanced_tile_fpr",
    "test_source_group_balanced_tile_fnr",
]
calibration_columns = [
    "calibration_source_group_balanced_tile_roc_auc",
    "calibration_object_roc_auc",
    "calibration_source_image_roc_auc",
    "calibration_source_group_balanced_tile_balanced_accuracy",
    "calibration_source_group_balanced_tile_fpr",
    "calibration_source_group_balanced_tile_fnr",
]
metric_columns = [column for column in test_columns if column in summary_df]
if not metric_columns:
    metric_columns = [
        column for column in calibration_columns if column in summary_df
    ]
display(
    summary_df[
        [
            "config_name",
            "seed",
            "selected_top_k_pixels",
            "selected_top_k_fraction",
            *metric_columns,
            "log_dir",
        ]
    ]
)

In [ ]:
display_row = summary_df.loc[
    (summary_df["config_name"] == DISPLAY_RUN[0])
    & (summary_df["seed"] == DISPLAY_RUN[1])
]
if not display_row.empty:
    run_dir = Path(display_row.iloc[0]["log_dir"])
    for name in (
        "calibration_training_curves.png",
        "calibration_top_k_sweep.png",
        "calibration_score_distribution.png",
    ):
        path = run_dir / name
        if path.is_file():
            display(NotebookImage(filename=str(path)))

In [ ]:
if not display_row.empty:
    run_dir = Path(display_row.iloc[0]["log_dir"])
    scores_path = run_dir / "test_scores.csv"
    metrics_path = run_dir / "test_metrics.json"
    if scores_path.is_file() and metrics_path.is_file():
        scores = pd.read_csv(scores_path)
        threshold = json.loads(metrics_path.read_text())["test_threshold"]
        scores["prediction"] = (scores["score"] >= threshold).astype(int)
        errors = scores.loc[scores["prediction"] != scores["label"]].copy()
        examples = pd.concat(
            [
                errors.loc[errors["label"] == 0].nlargest(1, "score"),
                errors.loc[errors["label"] == 1].nsmallest(1, "score"),
            ],
            ignore_index=True,
        )
        display(examples)
        for relative_path in examples["path"]:
            with Image.open(DATASET_ROOT / relative_path) as image:
                display(image.convert("RGB"))